# M and A, workshop 2 — Keys, duplicates and the values that lie

Last week you got three files loading correctly. They are still unusable: a gender column that
speaks six languages, ten spellings of one country, and an email address shared by 499 different
customers.

Today:

1. **Concat or merge?** — the question everyone answers wrong first.
2. **Normalise before you compare**, or your duplicates stay invisible.
3. **Decode a coded column from evidence**, instead of guessing.
4. **Check your key before you trust it.**

In [ ]:
# ABOUTME: M and A workshop 2 - merge keys, normalisation and deduplication on the real exports.
# ABOUTME: Includes decoding the 0/1 gender column from customers who appear in two systems.
import csv
import io
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

def qwasar_file(name):
    """Locate a dataset fetched by workshops/get_data.py, from wherever this notebook was opened."""
    for up in range(4):
        p = Path(*([".."] * up), "data", name)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"{name} is missing. Run `uv run get_data.py` in the workshops/ folder first - "
        "these notebooks only ever use the real datasets Qwasar provides.")

TABLE_1 = qwasar_file("only_wood_customer_us_1.csv")
TABLE_2 = qwasar_file("only_wood_customer_us_2.csv")
TABLE_3 = qwasar_file("only_wood_customer_us_3.csv")

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 20)
print("the three exports Only Wood handed over:")
for p in (TABLE_1, TABLE_2, TABLE_3):
    print(f"  {p.name}: {p.stat().st_size / 1000:.0f} kB, {sum(1 for _ in p.open()):,} lines")

In [ ]:
# The three loaders from workshop 1, so this notebook stands alone.
table_1 = pd.read_csv(TABLE_1)
table_2 = pd.read_csv(TABLE_2, sep=";", header=None, names=["age", "city", "gender", "name", "email"])
_header, _body = TABLE_3.read_text().split("\n", 1)
table_3 = pd.read_csv(io.StringIO(_body), sep="\t", header=None,
                      names=_header.strip().split(","), quoting=csv.QUOTE_NONE)
print(table_1.shape, table_2.shape, table_3.shape)

## 1. Concat, not merge

`pd.merge` joins tables **side by side**: same entities, different attributes. Customers and their
orders. That is not our situation.

All three files hold *the same kind of row* — a customer with a name, an email, a city. We stack
them **on top of each other** and then collapse the rows describing the same person. That is
`pd.concat`, followed by deduplication.

**Rule of thumb:** merge when you are adding columns; concat when you are adding rows. If you
cannot name the columns a join would add, you do not want a join.

## 2. Normalise before you compare

Two rows describe the same customer only if you can *see* that they do. Right now you cannot.

In [ ]:
def strip_types(s):
    """Table 3 wrote the type into the value: string_Male, "integer_26", boolean_1, character_M."""
    return (s.astype(str)
             .str.replace(r'^"|"$', "", regex=True)
             .str.replace(r"^(string|integer|character|boolean|float)_", "", regex=True))

def clean_text(s):
    """Stray quotes and backslashes survive from three generations of bad exports."""
    return (s.astype(str)
             .str.replace(r'[\\"]', "", regex=True)
             .str.strip()
             .replace({"": None, "nan": None}))

print("gender, table 1:", clean_text(table_1.Gender).value_counts().to_dict())
print("gender, table 3:", clean_text(strip_types(table_3.Gender)).value_counts().to_dict())
print()
print("country, table 3:", clean_text(strip_types(table_3.Country)).value_counts().head(10).to_dict())

**Six encodings for a gender. Ten spellings for one country.** And in table 1, a country recorded
as `12` — 1 682 times.

Before any comparison, every column needs **one** representation.

### Exercise 1 — normalise the country

`US`, `USA`, `U.S.A.`, `U.S.`, `Us`, `u.s.a.`, `United State Of America`, `United-State-Of-America`,
`united state of america`, `Usa` are one country. `12` and `1` are not countries at all.

Write `normalise_country`: every real spelling becomes `"United States of America"`, anything
unrecognisable becomes `None`. Do **not** invent a country for the junk.

Hint: lowercase, delete everything that is not a letter, then compare against a small set.

In [ ]:
def normalise_country(s):
    """Every spelling of the USA onto one string; unrecognised values become None."""
    v = clean_text(s)
    # TODO 1: build a comparison key - lowercase, letters only (so "U.S.A." and "Us" collide)
    # TODO 2: map keys that mean the USA to "United States of America", everything else to None
    return v

sample = pd.Series(["US", "USA", "U.S.A.", "united-state-of-america", "Us", "12", "1", None])
normalise_country(sample).tolist()

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

_out = normalise_country(sample)
check("all five spellings collapse to one", lambda: _out[:5].nunique() == 1)
check("that one value is the full name", lambda: _out.iloc[0] == "United States of America")
check("12 and 1 become None", lambda: _out.iloc[5] is None and _out.iloc[6] is None)

In [ ]:
before = clean_text(table_1.Country).nunique()
after = normalise_country(table_1.Country).nunique()
junk = int(normalise_country(table_1.Country).isna().sum())
print(f"table 1 country: {before} distinct spellings -> {after} after normalising")
print(f"rows left with no country at all: {junk:,} of {len(table_1):,}")

**The punchline:** `12` is not a country, and the honest output is "missing". Filling it with
"United States of America" because every other row says so is inventing data for thousands of
customers — and nobody reading your table later can tell which values you made up.

## 3. Decode the gender column from evidence

Tables 1, 2 and 3 all use `0` and `1` for some rows. Does `1` mean male or female?

You could guess. Or you could notice that **some customers appear in two files at once** —
explicit in one, coded in the other. Those records decide it.

In [ ]:
def gender_rows(df, email_col, gender_col, source):
    """One tidy frame of (email, gender, source), with emails comparable across files."""
    return pd.DataFrame({"email": clean_text(strip_types(df[email_col])).str.lower(),
                         "gender": clean_text(strip_types(df[gender_col])),
                         "source": source})

rows = pd.concat([gender_rows(table_1, "Email", "Gender", "table_1"),
                  gender_rows(table_2, "email", "gender", "table_2"),
                  gender_rows(table_3, "Email", "Gender", "table_3")], ignore_index=True)

# TODO 1: keep only rows whose email looks like a real address (a@b.tld)
# TODO 2: keep the emails that appear in more than one row
# TODO 3: keep the groups that mix a coded value (0/1) with an explicit one (Male/Female/M/F)
evidence = None
evidence

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("evidence found", lambda: evidence is not None and len(evidence) > 0)
check("three decisive customers", lambda: evidence.email.nunique() == 3)
check("every one of them pairs a 0 with Male",
      lambda: all(set(g) == {"0", "Male"} for _, g in evidence.groupby("email").gender))

Out of 30 000 rows, exactly **19 email addresses appear in more than one file**, and exactly
**three** of those pair a coded value with a written one:

| email | values |
|---|---|
| fadel.roberto@hotmail.com | `0`, `Male` |
| gerhold_keenan@hotmail.com | `Male`, `0` |
| howell_arron@yahoo.com | `0`, `Male` |

All three say the same thing: **`0` is male**, therefore `1` is female.

Three customers is thin evidence, and you should say so in your migration notes — but it is
*evidence*, and it beats the coin-flip that the alternative amounts to. Writing "0 = Male, based on
three customers present in two systems" is a sentence a reviewer can check. Writing "0 = Male"
with no justification is a sentence they cannot.

**Rule of thumb:** when a column's meaning is undocumented, look for records that appear twice.
Overlap is free documentation.

### Exercise 2 — normalise the gender

In [ ]:
def normalise_gender(s):
    """Six encodings down to two values. 0 = Male, 1 = Female (see the evidence above)."""
    v = clean_text(strip_types(s)).str.lower()
    # TODO: map the six spellings onto "Male" / "Female" (unmapped values become NaN for free)
    return v

g_sample = pd.Series(["Male", "M", "0", "Female", "F", "1", "boolean_1", "???"])
normalise_gender(g_sample).tolist()

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

_g = normalise_gender(g_sample)
check("the three male spellings agree", lambda: set(_g[:3]) == {"Male"})
check("the three female spellings agree", lambda: set(_g[3:6]) == {"Female"})
check("boolean_1 survives the type prefix", lambda: _g.iloc[6] == "Female")
check("nonsense becomes missing", lambda: pd.isna(_g.iloc[7]))

## 4. The key that is not a key

Email is the only identifier the three systems share, so it is the natural key for deduplication.
Look at it before trusting it.

In [ ]:
emails = pd.concat([clean_text(table_1.Email),
                    clean_text(table_2.email),
                    clean_text(strip_types(table_3.Email))], ignore_index=True).str.lower()
print("most frequent values:")
print(emails.value_counts().head(3).to_string())
print(f"\nrows with no email at all: {int(emails.isna().sum()):,}")

`forgottoask@woodinc` — **499 times**. It is not an address, it is a shrug: whoever typed the form
never asked for the email. It also has no dot after the `@`, so it is not even a valid address.

Deduplicate on it and you merge 499 unrelated customers into one person.

On top of that, **996 rows have no email at all** (489 of them in table 2). Those customers keep
their other data; they simply cannot be matched.

**Rule of thumb:** before a column becomes your key, check its most frequent values and its null
count. A key with a popular value is not a key.

### Exercise 3 — a valid-email filter

In [ ]:
def valid_email(s):
    """Lowercase real addresses; anything not shaped like a@b.tld becomes NaN."""
    v = clean_text(strip_types(s)).str.lower()
    # TODO: keep only values matching something@something.tld  (hint: .str.match and .where)
    return v

e_sample = pd.Series(["A.B@Gmail.com", "forgottoask@woodinc", "nope", None, "x@y.co"])
valid_email(e_sample).tolist()

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

_e = valid_email(e_sample)
check("real addresses are lowercased", lambda: _e.iloc[0] == "a.b@gmail.com")
check("the placeholder is rejected", lambda: pd.isna(_e.iloc[1]))
check("junk is rejected", lambda: pd.isna(_e.iloc[2]) and pd.isna(_e.iloc[3]))
check("a short TLD still passes", lambda: _e.iloc[4] == "x@y.co")

### Exercise 4 — how many customers did we actually buy?

Count the distinct customers under three policies, and watch the answer change.

1. **raw** — emails exactly as stored
2. **normalised** — type prefixes stripped, lowercased, trimmed
3. **validated** — normalised, with invalid addresses excluded

In [ ]:
raw_emails = pd.concat([table_1.Email, table_2.email, strip_types(table_3.Email)], ignore_index=True)

# TODO: fill the four counts
n_rows = len(raw_emails)
n_raw_distinct = None
n_normalised_distinct = None
n_valid_distinct = None

print(f"rows                : {n_rows:,}")
print(f"distinct raw        : {n_raw_distinct}")
print(f"distinct normalised : {n_normalised_distinct}")
print(f"distinct valid      : {n_valid_distinct}")

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("counts filled in", lambda: None not in (n_raw_distinct, n_normalised_distinct, n_valid_distinct))
check("normalising finds hidden duplicates", lambda: n_normalised_distinct < n_raw_distinct)
check("the placeholder is out of the valid count", lambda: n_valid_distinct < n_normalised_distinct)

In [ ]:
counts = {"rows": n_rows, "distinct\nraw": n_raw_distinct,
          "distinct\nnormalised": n_normalised_distinct, "distinct\nvalid": n_valid_distinct}
if any(v is None for v in counts.values()):
    print("Fill in the four counts above, then run this cell to see them side by side.")
else:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar(list(counts), list(counts.values()), color=["#888", "#c44", "#4a7", "#36c"])
    ax.bar_label(bars, fmt="%d")
    ax.set_ylabel("customers"); ax.set_title("How many customers did we actually buy?")
    plt.tight_layout()

Four different answers to one question, from the same three files:

- **30 000 rows** — what a naive `len(concat)` reports.
- **28 499 distinct raw emails** — already fewer, because some rows repeat exactly.
- **28 487 after normalising** — the extra twelve were hiding behind capitalisation and stray quotes.
- **28 486 valid** — and 1 495 rows now hold no usable key at all.

Only the last number is defensible, and you can only reach it by cleaning **before** counting.

## 5. Resolving a conflict

When two systems disagree about one customer, your options, in order of honesty:

1. **Keep the record from the more trustworthy source**, and write down which and why.
2. **Keep the first occurrence** — fine, as long as you admit it is arbitrary.
3. **Keep the most complete record** — fewest missing fields. Usually the best default when you
   have no reason to trust one system over another.
4. Average them. There is no average of `Male` and `F`. Do not.

Whatever you choose: it is a decision, it goes in the migration notes, and your reviewer will ask.

## What to take away

1. **Concat adds rows, merge adds columns.** Three customer lists is a concat.
2. **Normalise, then compare.** Duplicates differing by case or punctuation are invisible until you
   remove case and punctuation.
3. **Decode coded columns from overlapping records.** Three honest pairs beat a confident guess —
   and the sentence "based on three customers present in two systems" is checkable.
4. **Check the frequent values and the null count of your key.** `forgottoask@woodinc` would have
   merged 499 strangers into one customer.
5. **Missing is a legitimate value.** Inventing a country to avoid a `NaN` makes your table look
   complete and be wrong.

### Take it home
Apply all three normalisers to the three files and report two numbers: how many distinct customers
you end up with, and how many rows you could not key. Those are the first two lines of your
migration report.